# Иванов Матвей - подгруппа 1


### путь Б, учебная база
Семестровый проект про разбор партий в домино: там нужны выборки и сводки, а не многошаговые действия. Процедуру и функцию делаю на учебной базе `p3_student_demo`, свою схему не меняю.


### подготовка


In [13]:
# подключили библиотеки
import pymysql
import pandas as pd


In [14]:
# настроили соединение
conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="root",
    database="p3_student_demo",
    charset="utf8mb4",
    autocommit=True,
)


In [15]:
with conn.cursor() as cur:
    cur.execute("SELECT VERSION() AS version")
    row = cur.fetchone()
pd.DataFrame([row], columns=["version"])


,version
0,26.10.0


##### шаг 1
Учебные таблицы пересоздаю, как в разделе 0 памятки, и проверяю, что студенты и проекты на месте.


In [16]:
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS projects")
    cur.execute("DROP TABLE IF EXISTS students")
    cur.execute('''
        CREATE TABLE students (
            id INT PRIMARY KEY,
            name VARCHAR(100) NOT NULL
        )
    ''')
    cur.execute('''
        CREATE TABLE projects (
            id INT PRIMARY KEY,
            title VARCHAR(200) NOT NULL,
            student_id INT NOT NULL,
            created_at DATETIME NOT NULL,
            FOREIGN KEY (student_id) REFERENCES students(id)
        )
    ''')
    cur.execute('''
        INSERT INTO students (id, name) VALUES
        (1, 'Аня'), (2, 'Борис'), (3, 'Вера')
    ''')
    cur.execute('''
        INSERT INTO projects (id, title, student_id, created_at) VALUES
        (101, 'Распознавание изображений', 1, '2026-08-20 10:00:00'),
        (102, 'Учебный трекер', 1, '2026-09-03 14:30:00'),
        (103, 'Система рекомендаций', 2, '2026-09-15 09:15:00')
    ''')

def show(sql, args=None):
    with conn.cursor() as cur:
        cur.execute(sql, args)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

show("SELECT * FROM students")


,id,name
0,1,Аня
1,2,Борис
2,3,Вера


In [17]:
show("SELECT * FROM projects")


,id,title,student_id,created_at
0,101,Распознавание изображений,1,2026-08-20 10:00:00
1,102,Учебный трекер,1,2026-09-03 14:30:00
2,103,Система рекомендаций,2,2026-09-15 09:15:00


##### шаг 2. Процедура `practice_projects_since`
Два входа: id студента и дата. Показывает имя, название проекта и дату создания, только проекты этого студента не раньше переданной даты, от ранних к поздним.
Если дату не передали, срабатывает IF и процедура пишет «Дата не указана», не выполняя выборку.
`DELIMITER` здесь не нужен: весь `CREATE PROCEDURE` уходит на сервер одной командой.


In [18]:
with conn.cursor() as cur:
    cur.execute("DROP PROCEDURE IF EXISTS practice_projects_since")
    cur.execute('''
        CREATE PROCEDURE practice_projects_since(
            IN p_student_id INT,
            IN p_since DATETIME
        )
        BEGIN
            IF p_since IS NULL THEN
                SELECT 'Дата не указана' AS message;
            ELSE
                SELECT s.name, p.title, p.created_at
                FROM students AS s
                JOIN projects AS p ON p.student_id = s.id
                WHERE s.id = p_student_id
                  AND p.created_at >= p_since
                ORDER BY p.created_at;
            END IF;
        END
    ''')


##### проверка процедуры
Аня с 1 сентября: августовский проект отсекается, остаётся «Учебный трекер».
Вера с 1 сентября: проектов нет, строк 0.
Аня без даты: ветка IF, сообщение «Дата не указана».


In [19]:
show("CALL practice_projects_since(1, '2026-09-01 00:00:00')")


,name,title,created_at
0,Аня,Учебный трекер,2026-09-03 14:30:00


In [20]:
show("CALL practice_projects_since(3, '2026-09-01 00:00:00')")


,name,title,created_at


In [21]:
show('CALL practice_projects_since(1, NULL)')


,message
0,Дата не указана


##### шаг 3. Функция `practice_project_count_since`
Те же два параметра. Возвращает одно целое число: сколько проектов подходит, в том числе 0. Если даты нет, `RETURN NULL`.


In [22]:
with conn.cursor() as cur:
    cur.execute("DROP FUNCTION IF EXISTS practice_project_count_since")
    cur.execute('''
        CREATE FUNCTION practice_project_count_since(
            p_student_id INT,
            p_since DATETIME
        )
        RETURNS INT
        READS SQL DATA
        BEGIN
            DECLARE v_count INT DEFAULT 0;
            IF p_since IS NULL THEN
                RETURN NULL;
            END IF;
            SELECT COUNT(*) INTO v_count
            FROM projects
            WHERE student_id = p_student_id
              AND created_at >= p_since;
            RETURN v_count;
        END
    ''')


##### проверка функции
Аня с 1 сентября: 1. Аня с 1 октября: 0. Аня без даты: NULL.


In [23]:
show("SELECT practice_project_count_since(1, '2026-09-01 00:00:00') AS cnt")


,cnt
0,1


In [24]:
show("SELECT practice_project_count_since(1, '2026-10-01 00:00:00') AS cnt")


,cnt
0,0


In [25]:
show('SELECT practice_project_count_since(1, NULL) AS cnt')


,cnt
0,None


##### дополнительная проверка, дата 10 сентября 2026
Ожидание до запуска. У Ани проекты от 20 августа и 3 сентября, оба раньше 10 сентября: процедура вернёт 0 строк, функция — 0. У Бориса «Система рекомендаций» создана 15 сентября, она попадает в выборку: 1 строка и count 1.


In [26]:
show("CALL practice_projects_since(1, '2026-09-10 00:00:00')")


,name,title,created_at


In [27]:
show("CALL practice_projects_since(2, '2026-09-10 00:00:00')")


,name,title,created_at
0,Борис,Система рекомендаций,2026-09-15 09:15:00


In [28]:
show("""
SELECT
    practice_project_count_since(1, '2026-09-10 00:00:00') AS anya_cnt,
    practice_project_count_since(2, '2026-09-10 00:00:00') AS boris_cnt,
    practice_project_count_since(1, '2000-09-10 00:00:00') AS full_anya_cnt
""")


,anya_cnt,boris_cnt,full_anya_cnt
0,0,1,2


Ожидание совпало: у Ани пусто и 0, у Бориса одна строка «Система рекомендаций» и count 1. Проект Ани от 3 сентября не проходит границу «не раньше 10 сентября».


### нужны ли процедура и функция в семестровом проекте
Проект разбирает мои партии в домино и ищет, где теряются очки: по стартовой кости, порядку хода, формату и сопернику. Такой список уже даёт один SELECT с JOIN и WHERE, а процент очков и доля поражений считаются через GROUP BY и AVG. Процедура только сохранила бы тот же запрос под именем: отдельных шагов вроде проверки и записи она не добавляет и быстрее запрос не делает. Число партий с выбранной даты тоже считается обычным COUNT, без своей функции. Поэтому в проект я их не добавляю, а создавать и вызывать показываю на учебной базе.


In [29]:
conn.close()
